This is part 3 of the [Road to the Top](https://www.kaggle.com/code/jhoward/first-steps-road-to-the-top-part-1) series, in which I show the process I used to tackle the [Paddy Doctor](https://www.kaggle.com/competitions/paddy-disease-classification) competition, leading to four 1st place submissions. The previous notebook is available here: [part 2](https://www.kaggle.com/code/jhoward/first-steps-road-to-the-top-part-1).

**PW Note** 
I copied this from my working version of Road to the Top and removed all the experimenting steps that weren't necessary to actually submit. 

## Memory and gradient accumulation

First we'll repeat the steps we used last time to access the data and ensure all the latest libraries are installed, and we'll also grab the files we'll need for the test set:

In [ ]:
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
!(. "$HOME/.cargo/env"; pip install -Uqq fastkaggle fastai fastcore huggingface_hub "timm==0.6.13")

In [ ]:

from fastkaggle import *

comp = 'paddy-disease-classification'
path = setup_comp(comp)  # , install='fastai "timm>=0.6.2.dev0"')
from fastai.vision.all import *
set_seed(42)

tst_files = get_image_files(path/'test_images').sorted()

In [ ]:
df = pd.read_csv(path/'train.csv')


Now we'll set up a `train` function which is very similar to the steps we used for training in the last notebook. But there's a few significant differences...

The first is that I'm using a `finetune` argument to pick whether we are going to run the `fine_tune()` method, or the `fit_one_cycle()` method -- the latter is faster since it doesn't do an initial fine-tuning of the head. When we fine tune in this function I also have it calculate and return the TTA predictions on the test set, since later on we'll be ensembling the TTA results of a number of models. Note also that we no longer have `seed=42` in the `ImageDataLoaders` line -- that means we'll have different training and validation sets each time we call this. That's what we'll want for ensembling, since it means that each model will use slightly different data.

The more important change is that I've added an `accum` argument to implement *gradient accumulation*. As you'll see in the code below, this does two things:

1. Divide the batch size by `accum`
1. Add the `GradientAccumulation` callback, passing in `accum`.

In [ ]:
trn_path = path/'train_images'

In [ ]:
def train(arch, size, item=Resize(480, method='squish'), accum=1, finetune=True, epochs=12):
    dls = ImageDataLoaders.from_folder(trn_path, valid_pct=0.2, item_tfms=item,
        batch_tfms=aug_transforms(size=size, min_scale=0.75), bs=64//accum)
    cbs = GradientAccumulation(64) if accum else []
    learn = vision_learner(dls, arch, metrics=error_rate, cbs=cbs).to_fp16()
    if finetune:
        learn.fine_tune(epochs, 0.01)
        return learn.tta(dl=dls.test_dl(tst_files))
    else:
        learn.unfreeze()
        learn.fit_one_cycle(epochs, 0.01)

Let's create a function to find out how much memory it used, and also to then clear out the memory for the next run:

In [ ]:
import gc
def report_gpu():
    print(torch.cuda.list_gpu_processes())
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
report_gpu()

## Running the models

Using the previous notebook, I tried a bunch of different architectures and preprocessing approaches on small models, and picked a few which looked good. We'll using a `dict` to list our the preprocessing approaches we'll use for each architecture of interest based on that analysis:

In [ ]:
res = 640, 480

In [ ]:
models = {
    'convnext_large_in22k': {
        (Resize(res), (320, 224)),
    }, 'vit_large_patch16_224': {
        (Resize(480, method='squish'), 224),
        (Resize(res), 224),
    }, 'swinv2_large_window12_192_22k': {
        (Resize(480, method='squish'), 192),
        (Resize(res), 192),
    }, 'swin_large_patch4_window7_224': {
        (Resize(res), 224),
    }
}

Now we're ready to train all these models. Remember that each is using a different training and validation set, so the results aren't directly comparable.

We'll append each set of TTA predictions on the test set into a list called `tta_res`.

In [ ]:
import datetime

tta_res = []
report_gpu()

for i, (arch, details) in enumerate(models.items()):
    print(f"arch {i + 1} of {len(models)} of size {len(details)}")
    for j, (item, size) in enumerate(details):
        start = datetime.datetime.now()
        print('---', arch)
        print(size)
        print(item.name)
        # Epochs was 12, I do not have the patience!
        # tta_res.append(train(arch, size, item=item, accum=4, epochs=epochs))
        report_gpu()
        elapsed = datetime.datetime.now() - start
        print(f"Finished {i + 1}, {j + 1}" in {elapsed})
        print()


## Ensembling

Since this has taken quite a while to run, let's save the results, just in case something goes wrong!

In [ ]:
save_pickle('tta_res.pkl', tta_res)

`Learner.tta` returns predictions and targets for each rows. We just want the predictions:

In [ ]:
tta_prs = first(zip(*tta_res))

Originally I just used the above predictions, but later I realised in my experiments on smaller models that `vit` was a bit better than everything else, so I decided to give those double the weight in my ensemble. I did that by simply adding the to the list a second time (we could also do this by using a weighted average):

In [ ]:
tta_prs += tta_prs[1:3]

An *ensemble* simply refers to a model which is itself the result of combining a number of other models. The simplest way to do ensembling is to take the average of the predictions of each model:

In [ ]:
avg_pr = torch.stack(tta_prs).mean(0)
avg_pr.shape

That's all that's needed to create an ensemble! Finally, we copy the steps we used in the last notebook to create a submission file:

In [ ]:
dls = ImageDataLoaders.from_folder(
    trn_path, 
    valid_pct=0.2, 
    item_tfms=Resize(480, method='squish'),
    batch_tfms=aug_transforms(size=224, min_scale=0.75)
)

In [ ]:
idxs = avg_pr.argmax(dim=1)
vocab = np.array(dls.vocab)
ss = pd.read_csv(path/'sample_submission.csv')
ss['label'] = vocab[idxs]
ss.to_csv('submission.csv', index=False)

Now we can submit!